In [1]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt

In [2]:
#infilename = 'rates_muons_electrons_both_alphas_mx_10TeV_1000TeV_mxstep_10TeV.parquet'
#df = pd.read_parquet(infilename)

In [3]:
acceptance_filenames = ['acc_dm_model_core.parquet', 'acc_dm_model_floating.parquet', 'acc_dm_model_mono-energetic.parquet']

model_names = []
for f in acceptance_filenames:
    mname = f.split('dm_model_')[1].split('.')[0]
    print(mname)
    model_names.append(mname)


print(acceptance_filenames)
print(model_names)

core
floating
mono-energetic
['acc_dm_model_core.parquet', 'acc_dm_model_floating.parquet', 'acc_dm_model_mono-energetic.parquet']
['core', 'floating', 'mono-energetic']


In [5]:
# Prototype

dfacc = pd.read_parquet(acceptance_filenames[1])

dfacc

,M_DM,count_ecut10,org_nevents,frac_ecut10,count_hit_id_ecut10,frac_hit_id_ecut10,count_ecut100,frac_ecut100,count_hit_id_ecut100,frac_hit_id_ecut100,count_ecut1000,frac_ecut1000,count_hit_id_ecut1000,frac_hit_id_ecut1000,volume m3
0,1000.0,2182,2000000000,0.000001,42,2.100000e-08,1652,8.260000e-07,35,1.750000e-08,NaN,NaN,NaN,NaN,2.006598e+11
1,2000.0,3595,2000000000,0.000002,81,4.050000e-08,3080,1.540000e-06,69,3.450000e-08,145.0,7.250000e-08,2.0,1.000000e-09,2.006598e+11
2,3000.0,4519,2000000000,0.000002,102,5.100000e-08,3967,1.983500e-06,80,4.000000e-08,911.0,4.555000e-07,19.0,9.500000e-09,2.006598e+11
3,4000.0,5327,2000000000,0.000003,136,6.800000e-08,4792,2.396000e-06,121,6.050000e-08,1769.0,8.845000e-07,56.0,2.800000e-08,2.006598e+11
4,5000.0,6018,2000000000,0.000003,105,5.250000e-08,5538,2.769000e-06,95,4.750000e-08,2410.0,1.205000e-06,45.0,2.250000e-08,2.006598e+11
5,6000.0,6619,2000000000,0.000003,119,5.950000e-08,6127,3.063500e-06,108,5.400000e-08,2935.0,1.467500e-06,55.0,2.750000e-08,2.006598e+11
6,7000.0,6909,2000000000,0.000003,129,6.450000e-08,6458,3.229000e-06,117,5.850000e-08,3354.0,1.677000e-06,60.0,3.000000e-08,2.006598e+11
7,8000.0,7362,2000000000,0.000004,150,7.500000e-08,6881,3.440500e-06,138,6.900000e-08,3848.0,1.924000e-06,82.0,4.100000e-08,2.006598e+11
8,9000.0,7615,2000000000,0.000004,172,8.600000e-08,7160,3.580000e-06,164,8.200000e-08,4221.0,2.110500e-06,86.0,4.300000e-08,2.006598e+11
0,10000.0,8101,2000000000,0.000004,192,9.600000e-08,7639,3.819500e-06,186,9.300000e-08,4639.0,2.319500e-06,114.0,5.700000e-08,2.006598e+11


In [ ]:
dfacc['frac_ecut10'].iloc[0]

In [ ]:
#df.columns

In [ ]:
# From Claude

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────

import numpy as np
import pandas as pd




In [ ]:
# ── Cell 4: Merge with interpolation ─────────────────────────────────────────

def merge_acceptance(df1, df2, mass_col1="mx", mass_col2="M_DM", model_name=None):
    """
    Add frac_ecut* and volume* columns from df2 into df1,
    interpolating for df1 masses that fall between df2 grid points.

    Masses outside the df2 range are filled with NaN (with a warning).
    Exact mass matches use the df2 row directly.
    """
    # Columns to transfer
    accept_cols = [
        c for c in df2.columns
        if c.startswith("frac_ecut") or c.startswith("volume")
    ]
    if not accept_cols:
        raise ValueError("No frac_ecut* or volume* columns found in df2.")
    print(f"Transferring columns: {accept_cols}")

    # Sort df2 by mass
    df2_sorted = (
        df2[[mass_col2] + accept_cols]
        .sort_values(mass_col2)
        .reset_index(drop=True)
    )
    grid_masses = df2_sorted[mass_col2].values
    m1_vals     = df1[mass_col1].values

    # Warn about out-of-range masses
    '''
    out_of_range = (m1_vals < grid_masses.min()) | (m1_vals > grid_masses.max())
    if out_of_range.any():
        print(
            f"WARNING: {out_of_range.sum()} mass value(s) in df1 fall outside "
            f"the df2 range [{grid_masses.min():.4g}, {grid_masses.max():.4g}] "
            f"and will be set to NaN:\n  {m1_vals[out_of_range]}"
        )
    '''
    
    # Only flag masses below the grid minimum as out-of-range
    out_of_range = m1_vals < grid_masses.min()
    if out_of_range.any():
        print(
            f"WARNING: {out_of_range.sum()} mass value(s) in df1 fall below "
            f"the df2 minimum ({grid_masses.min():.4g}) "
            f"and will be set to NaN:\n  {m1_vals[out_of_range]}"
        )
    


    result = df1.copy()

    for col in accept_cols:
        y_grid = df2_sorted[col].values.astype(float)

        # np.interp: linear interpolation; out-of-range clamps to edge values,
        # which we then overwrite with NaN below
        #interp_vals = np.interp(m1_vals, grid_masses, y_grid)
        #interp_vals[out_of_range] = np.nan

        # Log-linear interpolation: linear in log(mass)
        log_grid   = np.log(grid_masses)
        #log_query  = np.log(np.clip(m1_vals, grid_masses.min(), grid_masses.max()))
        log_query = np.log(np.clip(m1_vals, grid_masses.min(), None))

        interp_vals = np.interp(log_query, log_grid, y_grid)
        interp_vals[out_of_range] = np.nan

        colname = f'{col}'
        if model_name is not None:
            colname = f'{col}_{model_name}'
        result[colname] = interp_vals

    return result

In [ ]:
# ── Cell 2: Load files ───────────────────────────────────────────────────────
infilename1 = 'rates_muons_electrons_both_alphas_mx_10TeV_1000TeV_mxstep_10TeV.parquet'
df1 = pd.read_parquet(infilename1)  # has column 'mx'

idx = 0
infilename2 = acceptance_filenames[idx]
model_name = model_names[idx]
df2 = pd.read_parquet(infilename2)  # has column 'M_DM'

print("df1 shape:", df1.shape)
print("df2 shape:", df2.shape)

In [ ]:
# ── Cell 3: Inspect ──────────────────────────────────────────────────────────

print("df1 masses:", sorted(df1["mx"].unique()))
print()
print("df2 masses:", sorted(df2["M_DM"].unique()))

In [ ]:
df_merged = merge_acceptance(df1, df2, model_name=model_name)


# ── Cell 5: Spot-check ───────────────────────────────────────────────────────

df_merged.head(10)

In [ ]:
df1['mx'].max()

In [ ]:
# Production
df1 = 1
df2 = 2
del df1
del df2

# ── Cell 2: Load files ───────────────────────────────────────────────────────
infilename1 = 'rates_muons_electrons_both_alphas_mx_10TeV_1000TeV_mxstep_10TeV.parquet'
df1 = pd.read_parquet(infilename1)  # has column 'mx'

#idx = 0

for idx in [0, 1, 2]:
    
    infilename2 = acceptance_filenames[idx]
    model_name = model_names[idx]
    df2 = pd.read_parquet(infilename2)  # has column 'M_DM'
    
    print("df1 shape:", df1.shape)
    print("df2 shape:", df2.shape)

    df_merged = merge_acceptance(df1, df2, model_name=model_name)

    del df1
    del df2

    df1 = df_merged

df1

In [ ]:
print(infilename1)

outfile = f'{infilename1.split(".parquet")[0]}_WITH_CALC_ACCEPTANCES.parquet'
print(outfile)

df1.to_parquet(outfile)